# Scaled Dot-Product Attention — Solution

Reference implementation for `01_scaled_dot_product_attention_exercise.ipynb`.

## Setup

In [1]:
import torch
import torch.nn as nn

## Solution 1 — `scaled_dot_product_attention`

In [2]:
def scaled_dot_product_attention(Q, K, V):
    """
    Args:
        Q: (..., n, d_k)
        K: (..., n, d_k)
        V: (..., n, d_v)
    Returns:
        output: (..., n, d_v)
    """
    #Q - bs,num_heads,seq_len,d_k
    #K - bs,num_heads,seq_len,d_k
    d_k = Q.shape[-1]

    attention_scores = torch.softmax(Q@K.transpose(-2,-1) / (d_k**0.5),dim=-1) ## bs,num_heads,seq_len,seq_len
    out = attention_scores@V #bs,num_heads,seq_len,d_k

    return out

**Why scale by `sqrt(d_k)`?** _Explain: dot products grow with `d_k`, peaking softmax → vanishing gradients. Scaling normalizes variance back to ~1._

## Solution 2 — `SelfAttention` module

In [3]:
class SelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.w_q = nn.Linear(d_model,d_model)
        self.w_k = nn.Linear(d_model,d_model)
        self.w_v = nn.Linear(d_model,d_model)

    def forward(self, x):
        # x -> bs,seq_len,d_model
        d_model = x.shape[-1]
        q = self.w_q(x)
        k = self.w_k(x)
        v = self.w_v(x)
        
        attention_scores = torch.softmax(q@k.transpose(-2,-1)/(d_model**0.5),dim=-1)
        out = attention_scores@v

        return out

## Run the tests

In [4]:
from tests import run_scaled_dot_product_tests
run_scaled_dot_product_tests(scaled_dot_product_attention, SelfAttention)

Running scaled dot-product attention...
  ✓ Output shape (4, 8)
  ✓ Batched: (2, 4, 8) input ok
  ✓ Uniform Q → identical output rows
  ✓ SelfAttention output shape correct
  ✓ Gradients flow through SelfAttention

  All 5 checks passed ✓



True